# CopyLane 판정 인코더 v11 — dev 행별 확률 CSV 뽑기 (소성민, 2026-10-07)

학습은 하지 않는다. **저장해 둔 v11 모델**을 불러 **v11 학습 때와 같은 dev 749행**에 추론만 돌려 CSV 로 낸다.

## ▶ 실행 방법

1. 메뉴 **런타임 → 모두 실행** (GPU 가 있으면 1분 안, CPU 여도 몇 분이면 끝난다)
2. Google Drive 연결 창이 뜨면 **허용**한다 — 아래 둘을 Drive 에서 찾아 온다.
   - 모델 zip — `내 드라이브/CopyLane_runs/v11_kcbert_*/copylane-encoder-kcbert-final_*.zip`
   - `golden.jsonl` — 공유 저장소(`CopyLane_store`)에서 sha `529c9705…` 로 찾는다
3. Drive 에서 못 찾으면 업로드 창이 뜬다 → 그 파일을 올린다 (`golden.jsonl` 은 `C:\SKN32-FINAL-3TEAM\data\derived\golden\golden.jsonl`).

## 나오는 파일 (자동으로 내려받아지고, Drive 의 그 실행 폴더에도 남는다)

| 파일 | 내용 |
|---|---|
| `dev_probs_<실행 이름>.csv` | 749행 · `id` + 유형별 확률 `p_<유형>` 아홉 칸 (문구 없음) |
| `dev_ids_v11_seed42.csv` | 749행 · `id` · `조건` · `bucket` |

- dev 분리는 v11 학습 노트북 3절을 그대로 옮겼다(원천별 묶음 15% · 시드 42). 뽑힌 id 목록의 지문을 미리 적어 두고 대조하므로, 다른 행이 뽑히면 멈춘다.
- 확률은 threshold 를 적용하기 전 값이다. threshold 와 유형 순서는 모델 zip 의 `label_scheme.json` 에 있다.
- 🔴 dev 749행에는 **조건 M 120행 · D 81행 · L 5행**이 들어 있다. 채점할 때 `dev_ids` 의 `조건` · `bucket` 칸으로 가른다.
- 문구는 싣지 않는다(D-303). id 로 골든과 맞춘다.


## 0. 설정 — 고정값 (바꾸지 않는다)

In [ ]:
BACKBONE = "kcbert"          # v11 은 KcBERT 만 돌렸다. kcelectra 모델 zip 이 생기면 이 값만 바꾼다
NOTEBOOK_VERSION = "v11"
MAX_LEN = 128
DEV_FRAC, DEV_SEED = 0.15, 42

# 재동결 10-05 판 (v10 · v11 학습과 같은 판)
EXPECT_GOLDEN = ("529c970556d77bd2c231526e20531294bf469f794e74a4796835a36f456eb7fe", 8383)
STORE_SHORTCUT_ID = "1_3HrAhrrLgy6jxAPprGGLjDQvpIsQ2h9"   # CopyLane_store (Drive 공유 저장소)

SEL = ["질병_예방치료_표방", "건강기능식품_오인", "의약품_오인", "거짓_과장", "소비자_기만", "후기_체험기_기만"]
CONFIRMED = SEL + ["부당_비교광고", "비방광고"]
AUX = ["기능성화장품_오인"]
LABEL_LIST = CONFIRMED + AUX
POSITIVE_CONDS, NONPOSITIVE_CONDS = ("C", "A", "B"), ("M", "D", "L")

# v11 학습 때의 dev — 결과 보고서(10-06)와 같은 수 · id 목록의 지문은 로컬 골든으로 10-07 에 뽑아 적었다
EXPECT_DEV_GROUPS, EXPECT_DEV_ROWS = 405, 749
EXPECT_DEV_BUCKETS = {"scored": 510, "pending": 32, "aux": 1, "neg": 5, "D": 81, "M": 120}
EXPECT_DEV_IDS_SHA = "c2a0e7a98e6e1bab73391ca1663266b4805a5c7cd47e91b11bc529400db91839"


## 1. 환경 · golden.jsonl · 모델 zip

In [ ]:
import os, sys, json, hashlib, shutil, math, random, time, glob, csv, zipfile, datetime
from collections import Counter, defaultdict
import numpy as np

try:
    from google.colab import files as colab_files, drive as colab_drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False
WORK = "/content" if IN_COLAB else os.getcwd()
os.chdir(WORK)

import torch, transformers
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} · transformers {transformers.__version__} · 장치 {torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU'}")

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

DRIVE_OK = False
if IN_COLAB:
    try:
        colab_drive.mount("/content/drive", force_remount=False)
        DRIVE_OK = os.path.isdir("/content/drive/MyDrive")
    except Exception as e:
        print("Drive 연결 안 됨 — 업로드로 받는다:", str(e)[:120])

# ── golden.jsonl — 이름이 아니라 내용(sha)으로 찾는다
def find_golden():
    want = EXPECT_GOLDEN[0]
    for p in sorted(glob.glob(os.path.join(WORK, "*.jsonl"))):
        if sha256(p) == want:
            return p, "작업 폴더"
    if DRIVE_OK:
        tail = f"copylane-derived/objects/{want[:2]}/{want}"
        roots = [f"/content/drive/.shortcut-targets-by-id/{STORE_SHORTCUT_ID}/CopyLane_store", "/content/drive/MyDrive/CopyLane_store"]
        roots += glob.glob("/content/drive/MyDrive/*/CopyLane_store") + glob.glob("/content/drive/Shareddrives/*/CopyLane_store")
        for r in roots:
            p = os.path.join(r, tail)
            if os.path.isfile(p) and sha256(p) == want:
                dst = os.path.join(WORK, "golden.jsonl"); shutil.copy(p, dst)
                return dst, "Drive 공유 저장소(CopyLane_store)"
    return None, None

GOLDEN_PATH, GOLDEN_SRC = find_golden()
if GOLDEN_PATH is None and IN_COLAB:
    print("⬆ 업로드 창에서 golden.jsonl 을 고른다")
    colab_files.upload()
    GOLDEN_PATH, GOLDEN_SRC = find_golden()
    if GOLDEN_PATH: GOLDEN_SRC = "직접 업로드"
if GOLDEN_PATH is None:
    raise SystemExit(f"🔴 sha {EXPECT_GOLDEN[0][:12]}… 인 golden.jsonl 을 찾지 못했다 — 재동결 10-05 판을 올린다")
print(f"✅ golden.jsonl sha {EXPECT_GOLDEN[0][:12]}…  ← {GOLDEN_SRC}")

# ── 모델 zip — Drive 의 v11 실행 폴더에서 찾는다 (여럿이면 가장 나중 실행)
ZIP_GLOB = f"copylane-encoder-{BACKBONE}-final*.zip"
def find_model_zip():
    c = sorted(glob.glob(os.path.join(WORK, ZIP_GLOB)))
    if c:
        return c[-1], "작업 폴더"
    if DRIVE_OK:
        c = sorted(glob.glob(f"/content/drive/MyDrive/CopyLane_runs/{NOTEBOOK_VERSION}_{BACKBONE}_*/{ZIP_GLOB}"))
        if c:
            return c[-1], "Drive"
    return None, None

MODEL_ZIP, MODEL_ZIP_SRC = find_model_zip()
if MODEL_ZIP is None and IN_COLAB:
    print(f"⬆ 업로드 창에서 v11 모델 zip({ZIP_GLOB}) 을 고른다 — 400MB 남짓이라 몇 분 걸린다")
    colab_files.upload()
    MODEL_ZIP, MODEL_ZIP_SRC = find_model_zip()
if MODEL_ZIP is None:
    raise SystemExit(f"🔴 v11 모델 zip 을 찾지 못했다 — 내 드라이브/CopyLane_runs/{NOTEBOOK_VERSION}_{BACKBONE}_*/ 를 확인한다")
print(f"✅ 모델 zip {MODEL_ZIP}  ← {MODEL_ZIP_SRC}")

EXTRACT = os.path.join(WORK, f"model_{NOTEBOOK_VERSION}_{BACKBONE}")
shutil.rmtree(EXTRACT, ignore_errors=True)
with zipfile.ZipFile(MODEL_ZIP) as z:
    z.extractall(EXTRACT)
_hits = glob.glob(os.path.join(EXTRACT, "**", "label_scheme.json"), recursive=True)
assert len(_hits) == 1, f"🔴 zip 안에 label_scheme.json 이 하나가 아니다: {_hits}"
MODEL_DIR = os.path.dirname(_hits[0])
SCHEME = json.load(open(_hits[0], encoding="utf-8"))
print("모델 폴더:", sorted(os.listdir(MODEL_DIR)))
print("실행 이름:", SCHEME.get("run_id"), "· 시드", SCHEME.get("seed"), "· best epoch", SCHEME.get("best_epoch"))

# ── 이 모델이 v11 · 같은 골든 판 · 같은 유형 순서인지
assert SCHEME.get("notebook") == NOTEBOOK_VERSION, f"🔴 v11 모델이 아니다: {SCHEME.get('notebook')}"
assert SCHEME.get("golden_sha256") == EXPECT_GOLDEN[0], "🔴 모델이 학습한 골든 판이 다르다"
assert SCHEME.get("label_list") == LABEL_LIST, f"🔴 유형 순서가 다르다: {SCHEME.get('label_list')}"
RUN_ID = SCHEME["run_id"]
TH = SCHEME["label_thresholds"]
print("threshold:", TH)


## 2. dev 분리 — v11 학습 때와 같은 749행

In [ ]:
def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

golden = load_jsonl(GOLDEN_PATH)
assert len(golden) == EXPECT_GOLDEN[1], f"행 수가 다르다: {len(golden)}"

def norm_text(s):
    return " ".join(s.split())

def group_key(r):
    rid = r["id"]
    if rid.startswith("inj:"):              # 'inj:T1:hf:0:0' → 원본 승인 문구 'hf:0:0'
        return rid.split(":", 2)[2]
    return rid.split("#")[0]

def normalize(r):
    cond = r.get("조건")
    labels = list(r.get("labels") or [])
    return {"id": r["id"], "group": group_key(r), "text": r["text"], "labels": labels,
            "target": [] if cond in NONPOSITIVE_CONDS else labels,
            "split": r["split"], "provenance": r.get("provenance"), "origin": r.get("origin"),
            "cond": cond, "unit": r.get("unit")}

all_rows = [normalize(r) for r in golden]
unknown = {l for r in all_rows for l in r["labels"]} - set(CONFIRMED) - set(AUX)
assert not unknown, f"🔴 예상 밖 라벨 — 라벨 목록이 바뀌었다: {unknown}"
assert {r["split"] for r in all_rows} == {"train", "test_sentence"}, {r["split"] for r in all_rows}

def bucket(r, types):
    if r["cond"] == "M": return "M"
    if r["cond"] == "D": return "D"
    if r["cond"] == "L": return "neg"
    if any(l not in types for l in r["labels"]): return "aux"
    if r["labels"]: return "scored"
    return "pending" if r["cond"] in POSITIVE_CONDS else "neg"

for r in all_rows:
    r["bucket"] = bucket(r, CONFIRMED)
train_all = [r for r in all_rows if r["split"] == "train"]

# ── 아래는 v11 학습 노트북 3절의 dev 분리를 글자 그대로 옮긴 것이다 (난수 소비 순서가 같아야 같은 행이 나온다)
def dev_eligible(r):
    return r["unit"] == "문장" and r["origin"] != "injected"

groups = defaultdict(list)
for r in train_all:
    groups[r["group"]].append(r)

prov_groups = defaultdict(list)
for g in sorted(groups):
    elig = [r for r in groups[g] if dev_eligible(r)]
    if elig:
        prov_groups[Counter(r["provenance"] for r in elig).most_common(1)[0][0]].append(g)
rng = random.Random(DEV_SEED)
dev_groups = set()
for prov in sorted(prov_groups):
    gs = prov_groups[prov][:]
    rng.shuffle(gs)
    dev_groups.update(gs[:max(1, math.ceil(DEV_FRAC * len(gs)))])

def conf_pos(rows, l):
    return sum(1 for r in rows if dev_eligible(r) and l in r["target"])
rest = [g for g in sorted(groups) if g not in dev_groups]
rng.shuffle(rest)
for l in SEL:
    total = conf_pos(train_all, l)
    want = min(10, math.ceil(DEV_FRAC * total))
    have = sum(conf_pos(groups[g], l) for g in dev_groups)
    for g in rest:
        if have >= want:
            break
        k = conf_pos(groups[g], l)
        if k and g not in dev_groups:
            dev_groups.add(g); have += k

dev_rows = [r for g in sorted(dev_groups) for r in groups[g] if dev_eligible(r)]
DEV_IDS = [r["id"] for r in dev_rows]
DEV_IDS_SHA = hashlib.sha256("\n".join(DEV_IDS).encode("utf-8")).hexdigest()
DEV_BUCKETS = dict(Counter(r["bucket"] for r in dev_rows))

print(f"dev: 묶음 {len(dev_groups)}개 · {len(dev_rows)}행 · 버킷 {DEV_BUCKETS}")
assert len(dev_groups) == EXPECT_DEV_GROUPS and len(dev_rows) == EXPECT_DEV_ROWS, "🔴 dev 행 수가 v11 학습 때와 다르다"
assert DEV_BUCKETS == EXPECT_DEV_BUCKETS, f"🔴 dev 버킷이 v11 학습 때와 다르다: {DEV_BUCKETS}"
assert DEV_IDS_SHA == EXPECT_DEV_IDS_SHA, f"🔴 dev id 목록이 다르다 — 지문 {DEV_IDS_SHA[:12]}…"
assert len(set(DEV_IDS)) == len(DEV_IDS), "🔴 dev id 가 겹친다"
print("✅ v11 학습 때와 같은 dev 749행이다 (id 지문 일치)")


## 3. 추론

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import average_precision_score

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(DEVICE)
assert model.config.num_labels == len(LABEL_LIST), model.config.num_labels
assert [model.config.id2label[i] for i in range(len(LABEL_LIST))] == LABEL_LIST, "🔴 모델 config 의 유형 순서가 다르다"
PAD_ID = tokenizer.pad_token_id

# ── 아래 세 함수는 v11 학습 노트북과 같다 (max_length 128 · 길이순 배치 · GPU 면 fp16)
def encode(rows):
    return tokenizer([r["text"] for r in rows], truncation=True, max_length=MAX_LEN)["input_ids"]

def batchify(ids_list, idx):
    m = max(len(ids_list[i]) for i in idx)
    ids = torch.full((len(idx), m), PAD_ID, dtype=torch.long)
    att = torch.zeros((len(idx), m), dtype=torch.long)
    for k, i in enumerate(idx):
        ids[k, :len(ids_list[i])] = torch.tensor(ids_list[i]); att[k, :len(ids_list[i])] = 1
    return ids.to(DEVICE), att.to(DEVICE)

@torch.no_grad()
def predict_probs(model, ids_list, bs=128):
    model.eval()
    out = []
    order = np.argsort([len(x) for x in ids_list])
    for s in range(0, len(order), bs):
        idx = order[s:s + bs]
        ids, att = batchify(ids_list, idx)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")):
            lg = model(input_ids=ids, attention_mask=att).logits
        out.append((idx, torch.sigmoid(lg.float()).cpu().numpy()))
    p = np.zeros((len(ids_list), model.config.num_labels), dtype=np.float32)
    for idx, v in out:
        p[idx] = v
    return p

t0 = time.time()
P_DEV = predict_probs(model, encode(dev_rows))
assert P_DEV.shape == (EXPECT_DEV_ROWS, len(LABEL_LIST)) and np.isfinite(P_DEV).all()
print(f"추론 끝 — {P_DEV.shape} · {time.time() - t0:.1f}초")

# ── 맞게 불렀는지 보는 참고 수치 — v11 결과 보고서(10-06)의 dev 값과 견준다
Y = np.zeros_like(P_DEV, dtype=int)
for i, r in enumerate(dev_rows):
    for l in r["target"]:
        Y[i, LABEL_LIST.index(l)] = 1
ap6 = [average_precision_score(Y[:, i], P_DEV[:, i]) for i in range(len(SEL))]
th = np.array([TH[l] for l in LABEL_LIST], dtype=np.float32)
flag = (P_DEV >= th).any(1)
pos = np.array([r["bucket"] in ("scored", "pending") for r in dev_rows])
print(f"dev macro PR-AUC (6종 · 749행 전체): {np.mean(ap6):.3f}   ← 보고서: 선택 시드 42 는 0.721")
print(f"dev 양성 탐지 재현율 (C·A·B · 8종 threshold): {int(flag[pos].sum())}/{int(pos.sum())}   ← 보고서: 473/542")
print("값이 보고서와 크게 다르면(PR-AUC 0.01 넘게) 다른 모델을 부른 것이다 — CSV 를 넘기지 않는다")


## 4. CSV 저장 · 내려받기

In [ ]:
OUT_DIR = os.path.join(WORK, "CopyLane_dev_export", RUN_ID)
os.makedirs(OUT_DIR, exist_ok=True)
PROBS_CSV = os.path.join(OUT_DIR, f"dev_probs_{RUN_ID}.csv")
IDS_CSV = os.path.join(OUT_DIR, f"dev_ids_{NOTEBOOK_VERSION}_seed{DEV_SEED}.csv")

with open(PROBS_CSV, "w", encoding="utf-8", newline="") as f:
    w = csv.writer(f)
    w.writerow(["id"] + [f"p_{l}" for l in LABEL_LIST])
    for r, p in zip(dev_rows, P_DEV):
        w.writerow([r["id"]] + [f"{float(x):.6f}" for x in p])
with open(IDS_CSV, "w", encoding="utf-8", newline="") as f:
    w = csv.writer(f)
    w.writerow(["id", "조건", "bucket"])
    for r in dev_rows:
        w.writerow([r["id"], r["cond"] or "", r["bucket"]])

# ── 다시 읽어 확인
with open(PROBS_CSV, encoding="utf-8", newline="") as f:
    back = list(csv.reader(f))
assert back[0] == ["id"] + [f"p_{l}" for l in LABEL_LIST] and len(back) == EXPECT_DEV_ROWS + 1
assert [row[0] for row in back[1:]] == DEV_IDS
print("열:", back[0])
print("첫 행:", back[1])
for p in (PROBS_CSV, IDS_CSV):
    print(f"  {os.path.basename(p):50s} {os.path.getsize(p) / 1e3:7.1f} KB")

if DRIVE_OK:
    try:
        dst = os.path.join("/content/drive/MyDrive/CopyLane_runs", RUN_ID)
        os.makedirs(dst, exist_ok=True)
        for p in (PROBS_CSV, IDS_CSV):
            shutil.copy(p, dst)
        print("Drive 보관:", dst)
    except Exception as e:
        print("Drive 보관 안 됨:", e)
if IN_COLAB:
    for p in (PROBS_CSV, IDS_CSV):
        colab_files.download(p); time.sleep(1)
    print("⬇ 내려받기를 시작했다 — 브라우저가 「여러 파일 다운로드」를 물으면 허용한다")
